<a href="https://colab.research.google.com/github/cristobalcresposandoval-lang/CSN/blob/main/datos_2_proyecto_1_parte_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import os

# Generar archivo de ejemplo si no existe
if not os.path.exists("ventas.csv"):
    print("Archivo 'ventas.csv' no encontrado. Generando un archivo de ventas de ejemplo...")
    np.random.seed(42)
    fechas = pd.date_range(start="2023-01-01", end="2023-12-31", freq="D")
    productos = ["Teclado", "Mouse", "Monitor", "Auriculares", "Laptop"]
    tiendas = ["Tienda Norte", "Tienda Sur", "Tienda Centro"]

    datos_ejemplo = {
        "Fecha": np.random.choice(fechas, size=200),
        "Producto": np.random.choice(productos, size=200),
        "Tienda": np.random.choice(tiendas, size=200),
        "Cantidad": np.random.randint(1, 10, size=200),
        "Precio_Unitario": np.random.choice([15.0, 25.0, 150.0, 45.0, 800.0], size=200)
    }
    df_ejemplo = pd.DataFrame(datos_ejemplo)
    # Introducir algunos duplicados y nulos para probar la fase de limpieza
    df_ejemplo = pd.concat([df_ejemplo, df_ejemplo.head(5)], ignore_index=True)
    df_ejemplo.loc[10:12, "Cantidad"] = np.nan

    df_ejemplo.to_csv("ventas.csv", index=False)
    print("Archivo 'ventas.csv' generado exitosamente.\n")

# Cargar dataset
df = pd.read_csv("ventas.csv")

# Limpieza de datos
df = df.drop_duplicates()
df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")
df["Cantidad"] = pd.to_numeric(df["Cantidad"], errors="coerce")
df["Precio_Unitario"] = pd.to_numeric(df["Precio_Unitario"], errors="coerce")

df = df.dropna(subset=["Fecha", "Cantidad", "Precio_Unitario", "Producto", "Tienda"])

# Transformación de datos
df["Ingreso_Total"] = df["Cantidad"] * df["Precio_Unitario"]
df["Mes"] = df["Fecha"].dt.month
df["Nombre_Mes"] = df["Fecha"].dt.month_name()
df["Año"] = df["Fecha"].dt.year

# Normalización de ventas
minimo = df["Ingreso_Total"].min()
maximo = df["Ingreso_Total"].max()

if maximo != minimo:
    df["Ventas_Normalizadas"] = (df["Ingreso_Total"] - minimo) / (maximo - minimo)
else:
    df["Ventas_Normalizadas"] = 0.0

# Clasificación de ventas
percentil_33 = df["Ingreso_Total"].quantile(0.33)
percentil_66 = df["Ingreso_Total"].quantile(0.66)

def clasificar_venta(ingreso):
    if ingreso <= percentil_33:
        return "Baja"
    elif ingreso <= percentil_66:
        return "Media"
    else:
        return "Alta"

df["Categoria_Venta"] = df["Ingreso_Total"].apply(clasificar_venta)

# Agrupación por producto y tienda
ventas_producto_tienda = df.groupby(["Producto", "Tienda"])["Ingreso_Total"].agg([
    "sum", "mean", "count", "min", "max", "std", "var"
])

print("Ventas por producto y tienda:")
display(ventas_producto_tienda)

# Agrupación por producto y mes
ventas_producto_mes = df.groupby(["Producto", "Año", "Mes"])["Ingreso_Total"].agg([
    "sum", "mean", "count", "min", "max", "std", "var"
])

print("Ventas por producto y mes:")
display(ventas_producto_mes)

# Agrupación por tienda y categoría
ventas_tienda_categoria = df.groupby(["Tienda", "Categoria_Venta"])["Ingreso_Total"].agg([
    "sum", "mean", "count", "min", "max", "std", "var"
])

print("Ventas por tienda y categoría:")
display(ventas_tienda_categoria)

# Función personalizada para calcular desviaciones
def calcular_desviacion(grupo):
    grupo = grupo.copy()
    media_grupo = grupo["Ingreso_Total"].mean()
    grupo["Media_Grupo"] = media_grupo
    grupo["Desviacion_Venta"] = grupo["Ingreso_Total"] - media_grupo
    return grupo

# Aplicar función por producto y tienda
df = df.groupby(["Producto", "Tienda"], group_keys=False).apply(calcular_desviacion)

print("Desviación de cada venta respecto a la media:")
display(df[["Producto", "Tienda", "Ingreso_Total", "Media_Grupo", "Desviacion_Venta"]].head(10))

# Resumen de ingresos por producto
resumen_productos = df.groupby("Producto")["Ingreso_Total"].sum().sort_values(ascending=False)

print("Ingresos por producto:")
display(resumen_productos)

print("Producto con mayores ingresos:")
print(resumen_productos.idxmax())

# Resumen de ingresos por tienda
resumen_tiendas = df.groupby("Tienda")["Ingreso_Total"].sum().sort_values(ascending=False)

print("Ingresos por tienda:")
display(resumen_tiendas)

print("Tienda con mayores ingresos:")
print(resumen_tiendas.idxmax())

# Estadísticas generales
print("Resumen de estadístico de ingresos:")
display(df["Ingreso_Total"].describe())

print("Varianza:", df["Ingreso_Total"].var())
print("Desviación estándar:", df["Ingreso_Total"].std())

# Exportar resultados
df.to_csv("ventas_parte_III_analizadas.csv", index=False)

ventas_producto_tienda.to_csv("ventas_por_producto_y_tienda.csv")
ventas_producto_mes.to_csv("ventas_por_producto_y_mes.csv")
ventas_tienda_categoria.to_csv("ventas_por_tienda_y_categoria.csv")

print("Análisis de la Parte III finalizado correctamente.")

Ventas por producto y tienda:


sum         mean  count    min     max  \
Producto    Tienda                                                      
Auriculares Tienda Centro   9125.0  1013.888889      9   45.0  4000.0   
            Tienda Norte   18050.0  1002.777778     18   30.0  6400.0   
            Tienda Sur     21670.0  1274.705882     17   60.0  5600.0   
Laptop      Tienda Centro   8365.0   643.461538     13   25.0  4000.0   
            Tienda Norte   16225.0  1352.083333     12   15.0  6400.0   
            Tienda Sur      5670.0  1417.500000      4  150.0  4800.0   
Monitor     Tienda Centro   7825.0   711.363636     11   45.0  4000.0   
            Tienda Norte   17505.0  1250.357143     14   45.0  7200.0   
            Tienda Sur     20275.0  1192.647059     17   25.0  6400.0   
Mouse       Tienda Centro   7285.0   662.272727     11   15.0  4000.0   
            Tienda Norte   13805.0  1061.923077     13   15.0  7200.0   
            Tienda Sur     11230.0   935.833333     12   75.0  6400.0   
Teclado     Tienda Centro  10650.0   887.500000     12   15.0  7200.0   
            Tienda Norte   10410.0   473.181818     22   25.0  5600.0   
            Tienda Sur     22480.0  1873.333333     12   15.0  6400.0   

                                   std           var  
Producto    Tienda                                    
Auriculares Tienda Centro  1244.108058  1.547805e+06  
            Tienda Norte   2003.437650  4.013762e+06  
            Tienda Sur     1545.816401  2.389548e+06  
Laptop      Tienda Centro  1105.053514  1.221143e+06  
            Tienda Norte   2170.271724  4.710079e+06  
            Tienda Sur     2258.367773  5.100225e+06  
Monitor     Tienda Centro  1149.282583  1.320850e+06  
            Tienda Norte   2282.706275  5.210748e+06  
            Tienda Sur     1966.832096  3.868428e+06  
Mouse       Tienda Centro  1182.293034  1.397817e+06  
            Tienda Norte   2056.839747  4.230590e+06  
            Tienda Sur     1753.063336  3.073231e+06  
Teclado     Tienda Centro  2033.042391  4.133261e+06  
            Tienda Norte   1172.849141  1.375575e+06  
            Tienda Sur     2767.697873  7.660152e+06

Ventas por producto y mes:


sum         mean  count     min     max  \
Producto    Año  Mes                                                
Auriculares 2023 1     1600.0  1600.000000      1  1600.0  1600.0   
                 2     7380.0   922.500000      8    45.0  4000.0   
                 3      300.0   150.000000      2   150.0   150.0   
                 4      120.0   120.000000      1   120.0   120.0   
                 5     1305.0   652.500000      2   405.0   900.0   
                 6     8020.0  1604.000000      5    30.0  6400.0   
                 7     3275.0   818.750000      4   225.0  1350.0   
                 8     3120.0   624.000000      5    90.0  1600.0   
                 9     7500.0  1500.000000      5    45.0  6400.0   
                 10    9900.0  3300.000000      3   300.0  5600.0   
                 11    2265.0   453.000000      5    45.0  1200.0   
                 12    4060.0  1353.333333      3    60.0  3200.0   
Laptop      2023 1     1945.0   648.333333      3    75.0  1600.0   
                 2       25.0    25.000000      1    25.0    25.0   
                 3      330.0   165.000000      2   150.0   180.0   
                 4      135.0    67.500000      2    45.0    90.0   
                 5     5725.0  1431.250000      4   175.0  4000.0   
                 6     6215.0   887.857143      7    15.0  4800.0   
                 7     4270.0  2135.000000      2   270.0  4000.0   
                 8     6535.0  3267.500000      2   135.0  6400.0   
                 9     4000.0  4000.000000      1  4000.0  4000.0   
                 10     675.0   337.500000      2   225.0   450.0   
                 11     315.0   157.500000      2    15.0   300.0   
                 12      90.0    90.000000      1    90.0    90.0   
Monitor     2023 1      100.0    50.000000      2    25.0    75.0   
                 2     2265.0   566.250000      4    45.0  1050.0   
                 3      125.0   125.000000      1   125.0   125.0   
                 4      810.0   405.000000      2   405.0   405.0   
                 5     2750.0   916.666667      3   125.0  2400.0   
                 6     5990.0  1996.666667      3    75.0  5600.0   
                 7      855.0   285.000000      3   180.0   405.0   
                 8     9695.0  1615.833333      6    45.0  7200.0   
                 9     6700.0  1340.000000      5    45.0  4000.0   
                 10    7135.0  1783.750000      4   150.0  6400.0   
                 11      60.0    60.000000      1    60.0    60.0   
                 12    9120.0  1140.000000      8    75.0  5600.0   
Mouse       2023 1      225.0   225.000000      1   225.0   225.0   
                 2     1055.0   211.000000      5    15.0   800.0   
                 3      900.0   450.000000      2   150.0   750.0   
                 4     1065.0   355.000000      3    75.0   900.0   
                 5      585.0   117.000000      5    75.0   225.0   
                 6      150.0   150.000000      1   150.0   150.0   
                 7      700.0   350.000000      2   100.0   600.0   
                 8    12750.0  2125.000000      6    15.0  7200.0   
                 9     6835.0  2278.333333      3   135.0  6400.0   
                 10    1425.0   712.500000      2    75.0  1350.0   
                 11    2450.0   816.666667      3   750.0   900.0   
                 12    4180.0  1393.333333      3    90.0  4000.0   
Teclado     2023 1     8550.0  1425.000000      6    15.0  7200.0   
                 2     1605.0   535.000000      3   105.0  1350.0   
                 3      355.0    88.750000      4    25.0   150.0   
                 4    12090.0  3022.500000      4    45.0  6400.0   
                 5      225.0   225.000000      1   225.0   225.0   
                 6      240.0   120.000000      2    15.0   225.0   
                 7      460.0   115.000000      4    30.0   180.0   
                 8     7005.0  1167.500000      6    45.0  6400.0   
                 9     7

Ventas por tienda y categoría:


sum         mean  count    min     max  \
Tienda        Categoria_Venta                                               
Tienda Centro Alta             37450.0  2080.555556     18  600.0  7200.0   
              Baja              1530.0    72.857143     21   15.0   135.0   
              Media             4270.0   251.176471     17  150.0   450.0   
Tienda Norte  Alta             66650.0  3507.894737     19  750.0  7200.0   
              Baja              2400.0    72.727273     33   15.0   135.0   
              Media             6945.0   257.222222     27  150.0   450.0   
Tienda Sur    Alta             76200.0  2627.586207     29  600.0  6400.0   
              Baja              1210.0    75.625000     16   15.0   125.0   
              Media             3915.0   230.294118     17  150.0   450.0   

                                       std           var  
Tienda        Categoria_Venta                             
Tienda Centro Alta             1796.658573  3.227982e+06  
              Baja               36.863648  1.358929e+03  
              Media             100.708884  1.014228e+04  
Tienda Norte  Alta             2504.621459  6.273129e+06  
              Baja               40.235458  1.618892e+03  
              Media              91.843652  8.435256e+03  
Tienda Sur    Alta             2272.175885  5.162783e+06  
              Baja               35.910769  1.289583e+03  
              Media             101.864963  1.037647e+04

... (6 outputs left out)

,Ingreso_Total
Tienda,
Tienda Sur,81325.0
Tienda Norte,75995.0
Tienda Centro,43250.0


Tienda con mayores ingresos:
Tienda Sur
Resumen de estadístico de ingresos:


,Ingreso_Total
count,197.000000
mean,1018.121827
std,1785.527569
min,15.000000
25%,90.000000
50%,225.000000
75%,900.000000
max,7200.000000


Varianza: 3188108.699368073
Desviación estándar: 1785.527568918518
Análisis de la Parte III finalizado correctamente.
